# Biohub random-hide: recalibrate ILP from visible GT

**Copy the code cells below into the end of your existing Kaggle pilot notebook and run them in the same live session.** Uploading this file as a separate Kaggle notebook normally creates a different `/kaggle/working` and will not see the previous outputs. Keep `/kaggle/working/biohub_random_hide` intact.

The cells reuse `visible.geff`, cached node/edge candidates, and the original image for visualization. They do not retrain, rerun inference, or use hidden GT to select costs. Original outputs stay intact. This is exploratory because the first hidden-GT result has already been inspected.

In [ ]:
from pathlib import Path
import json, subprocess, sys
from IPython.display import display, Image, FileLink
DATA_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
ROOT = Path('/kaggle/working/biohub_random_hide')
REPO = Path('/kaggle/working/linajea')
PINNED_CALIBRATION_COMMIT = None  # Optional SHA after pushing the calibration code
for required in (ROOT / 'visible.geff', ROOT / 'split.json',
                 ROOT / 'prediction' / 'candidate_nodes.csv',
                 ROOT / 'prediction' / 'candidate_edges.csv'):
    if not required.exists():
        raise FileNotFoundError(f'Missing previous-run output: {required}')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'biohub-io',
                    'https://github.com/hathanhv/linajea.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'biohub-io'], check=True)
commit = PINNED_CALIBRATION_COMMIT or subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'origin/biohub-io'],
    text=True).strip()
subprocess.run(['git', '-C', str(REPO), 'checkout', commit], check=True)
commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
(ROOT / 'calibration_code_commit.txt').write_text(commit + '\n')
print('Calibration code commit:', commit)
print('Candidate files are ready; checkpoint and maps will not be recomputed.')

In [ ]:
sample = json.loads((ROOT / 'split.json').read_text())['sample']
subprocess.run([sys.executable, '-m', 'run_scripts.biohub_random_hide', 'calibrate',
                '--data-dir', str(DATA_DIR), '--sample', sample,
                '--output-dir', str(ROOT)], cwd=REPO, check=True)
report = json.loads((ROOT / 'prediction_calibrated' / 'calibration.json').read_text())
print('Candidate-supported visible nodes/edges:',
      report['candidate_supported_visible_nodes'],
      report['candidate_supported_visible_edges'])
print('Chosen ILP costs and visible recovery:', report['chosen'])
print('Full grid:', ROOT / 'prediction_calibrated' / 'selection_grid.csv')

In [ ]:
subprocess.run([sys.executable, '-m', 'run_scripts.biohub_random_hide', 'evaluate',
                '--variant', 'calibrated', '--data-dir', str(DATA_DIR),
                '--sample', sample, '--output-dir', str(ROOT)],
               cwd=REPO, check=True)
metrics = json.loads((ROOT / 'evaluation_calibrated' / 'metrics.json').read_text())
print('Calibrated graph:', metrics['predicted_nodes'], 'nodes,',
      metrics['predicted_edges'], 'edges')
for threshold, result in metrics['thresholds_um'].items():
    print(threshold, 'um: hidden node', result['hidden_node_recovered'], '/',
          result['hidden_node_count'], '; hidden edge',
          result['hidden_edge_recovered'], '/', result['hidden_edge_count'],
          '; hidden-confirmed new prediction rate',
          result['hidden_confirmation_rate_lower_bound'])
print('These are exploratory hidden-GT results; unmatched predictions are unknown.')

In [ ]:
subprocess.run([sys.executable, '-m', 'run_scripts.biohub_random_hide', 'plot',
                '--variant', 'calibrated', '--data-dir', str(DATA_DIR),
                '--sample', sample, '--output-dir', str(ROOT)],
               cwd=REPO, check=True)
picture = ROOT / 'plots_calibrated' / 'comparison.png'
display(Image(filename=str(picture)))
print('Saved:', picture)
html = ROOT / 'plots_calibrated' / 'comparison_3d.html'
if html.exists():
    display(FileLink(str(html)))